In [ ]:
#| default_exp docsprocs

# docsprocs

> A docs-build notebook processor that color-codes cells by boopiter type, so the rendered site echoes the app's colored left bars.

Registered via `doc_procs` in `pyproject.toml [tool.nbdev]`, `color_cells` runs over every notebook during the docs build (before Quarto renders). It wraps each markdown/raw cell's source in a Quarto fenced div classed by its boopiter cell type (`.boop-note` green, `.boop-prompt` red, `.boop-raw` orange); `styles.css` turns those into the colored left bar. Code cells already carry `.cell-code`, so they're colored in CSS alone. The page's H1 title cell is left untouched so Quarto's title handling isn't disturbed.

In [ ]:
#| export
import re

# a Prompt+Reply markdown cell (solveit encoding) carries this separator -- see serialize.py
_SEP_RE = re.compile(r'##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_[0-9a-f]+ -->')

def color_cells(cell):
    "nbdev docs processor: wrap a markdown/raw cell's source in a Quarto fenced div classed by its boopiter cell type, so styles.css can draw boopiter's colored left bar (green note / red prompt / orange raw). Code cells carry `.cell-code` and are colored via CSS; the page-title (H1) cell is left alone."
    t = cell.get('cell_type'); src = cell.get('source') or ''
    if not src.strip(): return
    first = src.lstrip().splitlines()[0]
    if t == 'raw': cls = 'boop-raw'
    elif t == 'markdown':
        if first.startswith('# '): return          # leave the page-title (H1) cell alone
        cls = 'boop-prompt' if _SEP_RE.search(src) else 'boop-note'
    else: return                                    # code cells are handled in CSS
    cell['source'] = f'::: {{.boopcell .{cls}}}\n{src}\n:::'


In [ ]:
#| export
import hashlib, os, shutil, socket, subprocess, tempfile
from pathlib import Path

SHARE_REPO  = 'boops'                              # GitHub repo whose Pages site serves the rendered notebooks
SHARE_CLONE = Path.home()/'.boopiter'/'boops'      # local working clone, created on first share
# Assets the docs build already keeps next to its generated _quarto.yml. Copied as-is rather than
# regenerated: theme-toggle.html + styles.css are what give a page boopiter's sun/moon toggle and
# coloured cell bars, and they only stay in step with the docs site by being the same files.
_SHARE_ASSETS = ('_quarto.yml', 'nbdev.yml', 'styles.css', 'booptheme.scss', 'theme-toggle.html')

def _repo_root() -> Path:
    "The boopiter checkout this module was imported from -- where _proc/ and images/ live."
    return Path(__file__).parent.parent if '__file__' in globals() else Path.cwd()

def share_slug(path) -> str:
    "Six hex chars identifying a notebook by machine and location: sha256 of hostname + its canonical path. Deterministic, so re-sharing the same file overwrites the same page and a link you've already sent stays current -- that's the whole point of not using a random suffix. Keyed on more than the basename because 'example.ipynb' in two checkouts, or on two machines, are different documents that would otherwise fight over one URL. realpath (not abspath) so two symlinked routes to one file still collapse to a single page, and a NUL separator so host 'a' + path 'b/c' can't collide with host 'a/b' + path 'c'."
    key = f"{socket.gethostname()}\0{os.path.realpath(path)}"
    return hashlib.sha256(key.encode()).hexdigest()[:6]

def share_name(path) -> str:
    "The published page's directory name for a notebook: '<basename>_<slug>' (see share_slug)."
    return f'{Path(path).stem}_{share_slug(path)}'

def _share_quarto_config(proc:Path, out:Path) -> None:
    "Copy the docs build's generated Quarto config into `out` and adjust only what genuinely differs for a one-page render. Deliberately NOT a hand-written config: the theme, highlight styles, css and include-after-body all have to match the docs site, and the only way they stay matched is by being the same file. What changes: nbdev's pre/post-render steps (they build apilist/llms.txt for the full site), the sidebar and search (they index pages that aren't here), and sidebar.yml (generated per-site, absent from _proc). A favicon is added pointing at the logo the app itself serves -- Quarto's own option, so nothing has to be inlined by hand."
    import yaml
    cfg = yaml.safe_load((proc/'_quarto.yml').read_text())
    proj = cfg.setdefault('project', {})
    for k in ('pre-render', 'post-render', 'resources'): proj.pop(k, None)
    cfg['metadata-files'] = [f for f in cfg.get('metadata-files', []) if (proc/f).exists()]
    site = cfg.setdefault('website', {})
    site['sidebar'] = False
    site.setdefault('navbar', {})['search'] = False   # navbar stays: it carries Quarto's colour-scheme toggle
    logo = _repo_root()/'images'/'logo.png'
    if logo.exists():
        shutil.copy(logo, out/'logo.png')
        site['favicon'] = 'logo.png'
    (out/'_quarto.yml').write_text(yaml.safe_dump(cfg, sort_keys=False))

def render_share_html(nb_path, outdir=None) -> Path:
    "Render one notebook into a small Quarto site that looks like the docs, returning the directory to publish. Runs color_cells over the cells first -- the same processor the docs build uses via doc_procs -- then renders with the docs build's own config and assets (see _share_quarto_config), so the theme, coloured bars, sun/moon toggle and favicon are the ones already built for this project rather than second copies. Read and written with nbformat, NOT json.dump: nbformat stores a cell's source as a list of lines, and writing it back as one string makes Quarto treat the ::: fences as literal text, silently dropping every coloured bar. The notebook is rendered as index.ipynb so the published URL is a bare directory. `execute: enabled: false` is forced, so sharing a notebook never runs its code."
    import nbformat
    nb_path, proc = Path(nb_path), _repo_root()/'_proc'
    if not (proc/'_quarto.yml').exists():
        raise RuntimeError(f'{proc}/_quarto.yml not found -- run nbdev_docs once so the docs config exists')
    outdir = Path(outdir or tempfile.mkdtemp(prefix='boopshare-'))
    outdir.mkdir(parents=True, exist_ok=True)
    for a in _SHARE_ASSETS:
        if (proc/a).exists(): shutil.copy(proc/a, outdir/a)
    _share_quarto_config(proc, outdir)
    doc = nbformat.read(str(nb_path), as_version=4)
    for c in doc.cells: color_cells(c)
    nbformat.write(doc, str(outdir/'index.ipynb'))
    subprocess.run(['quarto', 'render', 'index.ipynb', '-M', 'execute:enabled:false'],
                   cwd=outdir, capture_output=True, timeout=600, check=True)
    site = next((p.parent for p in outdir.rglob('index.html')), None)
    if site is None: raise RuntimeError(f'quarto produced no index.html for {nb_path}')
    return site

def _gh_owner() -> str:
    "The GitHub login that owns the share repo, from the authenticated gh CLI."
    r = subprocess.run(['gh', 'api', 'user', '--jq', '.login'], capture_output=True, text=True, timeout=30, check=True)
    return r.stdout.strip()

def _ensure_share_clone(owner:str) -> Path:
    "The local clone of the share repo, cloned on first use and fast-forwarded after -- created (with Pages enabled) if the repo doesn't exist yet. Kept as a real clone rather than pushed from a temp dir each time so the history is coherent and a failed push can be retried."
    if not SHARE_CLONE.exists():
        SHARE_CLONE.parent.mkdir(parents=True, exist_ok=True)
        url = f'https://github.com/{owner}/{SHARE_REPO}.git'
        if subprocess.run(['gh', 'repo', 'view', f'{owner}/{SHARE_REPO}'], capture_output=True).returncode:
            subprocess.run(['gh', 'repo', 'create', f'{owner}/{SHARE_REPO}', '--public',
                            '-d', 'Notebooks shared from boopiter'], capture_output=True, timeout=60, check=True)
            subprocess.run(['git', 'init', '-q', '-b', 'main', str(SHARE_CLONE)], check=True, timeout=60)
            (SHARE_CLONE/'README.md').write_text('Notebooks shared from boopiter, served via GitHub Pages.\n')
            for c in (['git','add','-A'], ['git','commit','-qm','init'], ['git','remote','add','origin',url],
                      ['git','push','-q','-u','origin','main']):
                subprocess.run(c, cwd=SHARE_CLONE, check=True, timeout=120)
            subprocess.run(['gh','api','-X','POST',f'repos/{owner}/{SHARE_REPO}/pages',
                            '-f','source[branch]=main','-f','source[path]=/'], capture_output=True, timeout=60)
        else:
            subprocess.run(['git', 'clone', '-q', url, str(SHARE_CLONE)], check=True, timeout=180)
    else:
        subprocess.run(['git', 'pull', '-q', '--ff-only'], cwd=SHARE_CLONE, capture_output=True, timeout=120)
    return SHARE_CLONE

def share_notebook(nb_path) -> str:
    "Render `nb_path` and publish it to the share repo's Pages site, returning the public URL. Overwrites in place: the directory is named from the notebook's identity (see share_slug), so re-sharing after a correction updates the page a recipient already has the link to, rather than minting a new URL. The old copy is removed first, so a file that disappears from a re-render doesn't linger. Returns as soon as the push lands -- GitHub Pages then takes roughly a minute to rebuild, so the URL 404s briefly before going live. Raises if quarto, gh or git fail; the caller surfaces that (see share_nb in cells.py)."
    site  = render_share_html(nb_path)
    owner = _gh_owner()
    clone = _ensure_share_clone(owner)
    dest  = clone/share_name(nb_path)
    if dest.exists(): shutil.rmtree(dest)
    shutil.copytree(site, dest)
    subprocess.run(['git', 'add', '-A'], cwd=clone, check=True, timeout=60)
    if subprocess.run(['git', 'diff', '--cached', '--quiet'], cwd=clone).returncode:  # nothing staged == unchanged notebook
        subprocess.run(['git', 'commit', '-qm', f'Share {dest.name}'], cwd=clone, check=True, timeout=60)
        subprocess.run(['git', 'push', '-q'], cwd=clone, check=True, timeout=180)
    return f'https://{owner}.github.io/{SHARE_REPO}/{dest.name}/'